# Imports

In [1]:
from cyvcf2 import VCF
import pandas as pd
import numpy as np
from scipy import stats

# Load Data

In [2]:
vcf = VCF("data/ps3_gwas.vcf.gz")

In [3]:
phen = pd.read_csv("data/ps3_gwas.phen", sep="\t", header=None)

# Clean and Align Phenotype Data

In [4]:
phen.columns = ["FID","IID","PHEN"]
phen = phen.set_index("IID")
samples = vcf.samples
y = phen.loc[samples]["PHEN"].values

# GWAS Loop

In [5]:
# Set up output file
with open("gwas_results.tsv", "w") as out:
    out.write("CHR\tBP\tBETA\tP\tN\n")

    max_snps = 50
    count = 0

    for v in vcf:
        # if count >= max_snps:
        #     break

        # Convert genotype to dosage-like numeric (0/1/2); missing -> nan
        g = v.gt_types.astype(float)
        g[g == 3] = np.nan

        # Per-SNP complete-case mask
        mask = ~np.isnan(g) & ~np.isnan(y)
        g2 = g[mask]
        y2 = y[mask]

        # MAF on non-missing genotypes
        maf = min(g2.mean() / 2, 1 - g2.mean() / 2)
        if maf < 0.01:
            continue

        # Regression on filtered samples
        y0 = y2 - y2.mean()
        g0 = g2 - g2.mean()
        df = len(y2) - 2

        beta = (g0 @ y0) / (g0 @ g0)

        resid = y0 - beta * g0
        se = np.sqrt((resid @ resid) / df / (g0 @ g0))

        t = beta / se
        p = 2 * stats.t.sf(abs(t), df)

        out.write(f"{v.CHROM}\t{v.POS}\t{beta}\t{p}\t{len(y2)}\n")
        count += 1

/var/folders/hm/5xy_pph577gfc54kqp3v22980000gn/T/ipykernel_87283/214310404.py:22: RuntimeWarning: Mean of empty slice
  maf = min(g2.mean() / 2, 1 - g2.mean() / 2)
/var/folders/hm/5xy_pph577gfc54kqp3v22980000gn/T/ipykernel_87283/214310404.py:27: RuntimeWarning: Mean of empty slice
  y0 = y2 - y2.mean()
/var/folders/hm/5xy_pph577gfc54kqp3v22980000gn/T/ipykernel_87283/214310404.py:28: RuntimeWarning: Mean of empty slice
  g0 = g2 - g2.mean()
/var/folders/hm/5xy_pph577gfc54kqp3v22980000gn/T/ipykernel_87283/214310404.py:34: RuntimeWarning: divide by zero encountered in scalar divide
  se = np.sqrt((resid @ resid) / df / (g0 @ g0))
